# 01e - Grid de Augmentation (fase 2)

Treinamos com **augmentation aleatória** (cada imagem recebe, com prob `p_apply`, um método e valor sorteados de um pool) e avaliamos em ArtiFact **limpo** — teste de robustez cross-generator (Wang et al. 2020).

**Grid sobre as combinações de métodos** (espaço pequeno e discreto → enumerar garante cobertura e é robusto a ruído). 15 combinações × 2 seeds.

Classes de augmentation vêm de `aug_utils.py` (módulo importável) → permite `num_workers>0` no Windows.

- **Pool**: subconjuntos não-vazios de `{jpeg, blur, downscale, noise}` → 15 combinações
- **Faixas** (região leve do sweep): jpeg 50-90, blur 0-1.5, downscale 1-2, noise 0-0.05
- **Eval**: ArtiFact limpo, fixo e balanceado

Baselines: **raw=0.632** | **fixo (jpeg q70)=0.678**.

In [ ]:
import sys
import json
import time
import random
from itertools import combinations
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models
from sklearn.metrics import roc_auc_score
from tqdm.notebook import tqdm

sys.path.insert(0, str(Path.cwd()))
from aug_utils import RandomAugment, PathListDataset, clean_transform, train_transform

In [ ]:
PROJECT_ROOT    = Path.cwd().resolve().parent
_data_root_file = PROJECT_ROOT / "data_root.env"
DATA_ROOT       = Path(_data_root_file.read_text().strip()) if _data_root_file.exists() else PROJECT_ROOT / "data"

RAW_DIR       = DATA_ROOT / "raw" / "140k_faces" / "real_vs_fake" / "real-vs-fake"
ARTIFACT_DIR  = DATA_ROOT / "raw" / "artifact_faces"
RESULTS_DIR   = PROJECT_ROOT / "artifacts" / "aug_grid"
FIGURES_DIR   = PROJECT_ROOT / "reports" / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH  = RESULTS_DIR / "aug_grid_results.json"

IMAGE_SIZE      = 224
BATCH_SIZE      = 32
NUM_WORKERS     = 4           # workers OK: classes vêm de aug_utils.py
NUM_EPOCHS      = 10
SAMPLE_FRACTION = 0.05
ARTIFACT_N_PER_CLASS = 2000
ARTIFACT_EVAL_SEED   = 42

POOL_METHODS = ["jpeg", "blur", "downscale", "noise"]
ALL_POOLS = [list(x) for r in range(1, len(POOL_METHODS)+1) for x in combinations(POOL_METHODS, r)]
P_APPLY = 0.6
SEEDS   = [42, 123]
AUG_RANGES = {"jpeg": (50, 90), "blur": (0.0, 1.5), "downscale": (1.0, 2.0), "noise": (0.0, 0.05)}

BASELINES = {"raw": 0.632, "fixo (jpeg q70)": 0.678}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE, "| num_workers:", NUM_WORKERS)
print(f"Combinações: {len(ALL_POOLS)} | seeds: {len(SEEDS)} | total: {len(ALL_POOLS)*len(SEEDS)}")
print("Baselines:", BASELINES)

## 1. Dados (ArtiFact limpo, fixo e balanceado)

In [ ]:
def _list(folder):
    fs = []
    for e in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
        fs += list(folder.glob(e))
    return sorted(fs)

CLEAN_TF = clean_transform(IMAGE_SIZE)

_rng = random.Random(ARTIFACT_EVAL_SEED)
_real = _rng.sample(_list(ARTIFACT_DIR / "real"), ARTIFACT_N_PER_CLASS)
_fake = _rng.sample(_list(ARTIFACT_DIR / "fake"), ARTIFACT_N_PER_CLASS)
ARTIFACT_ITEMS = [(p, 1) for p in _real] + [(p, 0) for p in _fake]
artifact_loader = DataLoader(PathListDataset(ARTIFACT_ITEMS, CLEAN_TF),
                             batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f"ArtiFact eval (limpo): {len(_real)} real + {len(_fake)} fake")

def sample_subset(dataset, fraction, seed):
    n = int(len(dataset) * fraction)
    idx = random.Random(seed).sample(range(len(dataset)), n)
    return Subset(dataset, idx)

def build_train_valid(pool, p_apply, seed):
    aug = RandomAugment(pool, p_apply, AUG_RANGES)
    train_tf = train_transform(IMAGE_SIZE, aug)
    train_ds = sample_subset(datasets.ImageFolder(RAW_DIR / "train", transform=train_tf), SAMPLE_FRACTION, seed)
    valid_ds = sample_subset(datasets.ImageFolder(RAW_DIR / "valid", transform=CLEAN_TF), SAMPLE_FRACTION, seed)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
    valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    return train_loader, valid_loader

print("Loaders definidos.")

## 2. Treino + Avaliação

In [ ]:
def build_model(seed=42):
    torch.manual_seed(seed)
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    in_f = model.fc.in_features
    model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(in_f, 2))
    return model.to(DEVICE)

@torch.no_grad()
def auc_on(model, loader):
    model.eval()
    yt, yp = [], []
    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE)
        p = torch.softmax(model(imgs), dim=1)[:, 1].cpu().numpy()
        yp.extend(p.tolist()); yt.extend(lbls.tolist())
    return float(roc_auc_score(yt, yp))

def run_trial(pool, p_apply, seed):
    train_loader, valid_loader = build_train_valid(pool, p_apply, seed)
    model = build_model(seed)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    for _ in range(NUM_EPOCHS):
        model.train()
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), lbls)
            loss.backward()
            optimizer.step()
    val_auc      = auc_on(model, valid_loader)
    artifact_auc = auc_on(model, artifact_loader)
    del model; torch.cuda.empty_cache()
    return val_auc, artifact_auc

print("Funções de treino definidas.")

## 3. Grid das 15 combinações\n\nSalva incrementalmente — retoma de onde parou.

In [ ]:
if RESULTS_PATH.exists():
    results = json.loads(RESULTS_PATH.read_text())
    done = {(r["pool"], r["seed"]) for r in results}
    print(f"Retomando: {len(results)} runs já feitos.")
else:
    results = []
    done = set()

jobs = [(pool, seed) for pool in ALL_POOLS for seed in SEEDS
        if ("+".join(pool), seed) not in done]
print(f"Runs restantes: {len(jobs)}")

for pool, seed in tqdm(jobs, desc="Grid augmentation"):
    start = time.time()
    val_auc, artifact_auc = run_trial(pool, P_APPLY, seed)
    elapsed = time.time() - start
    results.append({"pool": "+".join(pool), "n_methods": len(pool), "p_apply": P_APPLY, "seed": seed,
                    "val_auc_140k": round(val_auc, 4), "artifact_auc": round(artifact_auc, 4), "time": round(elapsed, 1)})
    RESULTS_PATH.write_text(json.dumps(results, indent=2))
    tqdm.write(f"{'+'.join(pool):<28} seed={seed} | 140k={val_auc:.3f} | artifact={artifact_auc:.3f} | {elapsed:.0f}s")

print("Grid concluído.")

## 4. Resultados — AUC por combinação (média dos seeds)

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
agg = (df.groupby("pool")
         .agg(artifact_auc=("artifact_auc", "mean"), std=("artifact_auc", "std"), n_methods=("n_methods", "first"))
         .reset_index().sort_values("artifact_auc", ascending=False))
agg["std"] = agg["std"].fillna(0.0)
print("Combinações por AUC cross-generator (ArtiFact limpo):\n")
print(agg.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
best = agg.iloc[0]
print(f"\n→ Melhor combinação: [{best['pool']}] | AUC = {best['artifact_auc']:.4f} ± {best['std']:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
order = agg.sort_values("artifact_auc")
colors = plt.cm.viridis((order["n_methods"] - 1) / 3)
ax.barh(order["pool"], order["artifact_auc"], xerr=order["std"], color=colors, alpha=0.85, capsize=3)
ax.axvline(BASELINES["raw"], color="black", linestyle="--", label=f"raw ({BASELINES['raw']:.3f})")
ax.axvline(BASELINES["fixo (jpeg q70)"], color="tomato", linestyle="--", label=f"melhor fixo ({BASELINES['fixo (jpeg q70)']:.3f})")
ax.set_xlabel("AUC ArtiFact (cross-generator, limpo)")
ax.set_title("Grid de Augmentation — combinações (cor = nº de métodos)")
ax.legend(); ax.grid(alpha=0.3, axis="x")
ax.set_xlim(0.55, max(0.75, order["artifact_auc"].max() + 0.03))
plt.tight_layout()
plt.savefig(FIGURES_DIR / "aug_grid_results.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Batem raw (0.632): {(agg['artifact_auc']>BASELINES['raw']).sum()}/{len(agg)}")
print(f"Batem fixo (0.678): {(agg['artifact_auc']>BASELINES['fixo (jpeg q70)']).sum()}/{len(agg)}")

## 5. Desempate — top combinações com mais seeds

Re-roda as **top 3** com **5 seeds** (2 do grid + 3 novos) para escolher o vencedor com confiança.

In [ ]:
TIEBREAK_TOP_N = 3
TIEBREAK_SEEDS = [42, 123, 7, 2024, 99]
TIEBREAK_PATH  = RESULTS_DIR / "tiebreak_results.json"

top_pools = agg.head(TIEBREAK_TOP_N)["pool"].tolist()
print("Top combinações em desempate:", top_pools)
grid_by_key = {(r["pool"], r["seed"]): r["artifact_auc"] for r in results}
tb = json.loads(TIEBREAK_PATH.read_text()) if TIEBREAK_PATH.exists() else []
tb_done = {(r["pool"], r["seed"]) for r in tb}

tb_jobs = []
for pool_str in top_pools:
    pool = pool_str.split("+")
    for seed in TIEBREAK_SEEDS:
        if (pool_str, seed) in grid_by_key or (pool_str, seed) in tb_done:
            continue
        tb_jobs.append((pool, pool_str, seed))
print(f"Runs novos no desempate: {len(tb_jobs)}")

for pool, pool_str, seed in tqdm(tb_jobs, desc="Desempate"):
    start = time.time()
    val_auc, artifact_auc = run_trial(pool, P_APPLY, seed)
    elapsed = time.time() - start
    tb.append({"pool": pool_str, "seed": seed, "val_auc_140k": round(val_auc, 4),
               "artifact_auc": round(artifact_auc, 4), "time": round(elapsed, 1)})
    TIEBREAK_PATH.write_text(json.dumps(tb, indent=2))
    tqdm.write(f"{pool_str:<28} seed={seed} | artifact={artifact_auc:.3f} | {elapsed:.0f}s")
print("Desempate concluído.")

In [ ]:
tb_by_key = {(r["pool"], r["seed"]): r["artifact_auc"] for r in tb}
rows = []
for pool_str in top_pools:
    aucs = []
    for seed in TIEBREAK_SEEDS:
        a = grid_by_key.get((pool_str, seed))
        if a is None:
            a = tb_by_key.get((pool_str, seed))
        if a is not None:
            aucs.append(a)
    rows.append({"pool": pool_str, "n_seeds": len(aucs), "artifact_auc": float(np.mean(aucs)),
                 "std": float(np.std(aucs, ddof=1)) if len(aucs) > 1 else 0.0,
                 "min": float(np.min(aucs)), "max": float(np.max(aucs))})

final = pd.DataFrame(rows).sort_values("artifact_auc", ascending=False).reset_index(drop=True)
print("Desempate (5 seeds):\n")
print(final.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
champ = final.iloc[0]
print(f"\n→ VENCEDOR: [{champ['pool']}] | AUC = {champ['artifact_auc']:.4f} ± {champ['std']:.4f} (n={champ['n_seeds']})")
print(f"   vs raw={BASELINES['raw']:.3f} (+{champ['artifact_auc']-BASELINES['raw']:.3f})")

fig, ax = plt.subplots(figsize=(8, 4))
fo = final.sort_values("artifact_auc")
ax.barh(fo["pool"], fo["artifact_auc"], xerr=fo["std"], color="seagreen", alpha=0.85, capsize=4)
ax.axvline(BASELINES["raw"], color="black", linestyle="--", label=f"raw ({BASELINES['raw']:.3f})")
ax.axvline(BASELINES["fixo (jpeg q70)"], color="tomato", linestyle="--", label=f"melhor fixo ({BASELINES['fixo (jpeg q70)']:.3f})")
ax.set_xlabel(f"AUC ArtiFact (limpo) — {len(TIEBREAK_SEEDS)} seeds")
ax.set_title("Desempate das top combinações")
ax.legend(); ax.grid(alpha=0.3, axis="x")
ax.set_xlim(0.60, max(0.75, fo["max"].max() + 0.02))
plt.tight_layout()
plt.savefig(FIGURES_DIR / "aug_grid_tiebreak.png", dpi=150, bbox_inches="tight")
plt.show()
final.to_json(RESULTS_DIR / "tiebreak_summary.json", orient="records", indent=2)
print("Resumo salvo em:", RESULTS_DIR / "tiebreak_summary.json")